# Gmail corpus pilot — one mailroom-dataset document fired through the agent mailbox

**Honesty label: OFFLINE by default.** Nothing is sent and no network call happens unless
you explicitly flip the `FIRE` interlock (cell 2) — and the document text always comes from
the **mailroom-dataset HuggingFace dataset** (`Lucius-Morningstar/mailroom-dataset`, schema v9):
offline from the committed, sha-verified snapshot, or live from the Hub behind the
`NB-OPT-IN-NETWORK` marker below via the canonical `pipeline/hf_corpus_loader`.

**What one FIRE does** (a single-document upload — the free triage lane):

1. sends ONE attachment (a corpus document) to the agent mailbox — real SMTP, or a
   network-free `mock` drop that uses the poller's exact inbox+sidecar shape;
2. the live watcher claims it, the free triage team classifies + extracts, the document
   archives (or parks) with an auditable trail, the completion echo replies;
3. this notebook then verifies the run: terminal manifest, catalog row, audit hash-chain,
   relations edges, echo events — and compares the pipeline's output against the corpus
   row's ground truth (`expected`, `expected_subclass`, `expected_stage`, insurance fields);
4. writes `data/pilot_runs/<stamp>_<token>/report.{json,md}` — the durable pilot log.

**Prerequisites:** the watcher running (`PYTHONPATH=src python -m pipeline.watcher`, plus
the watchdog), `.env` with `GMAIL_ADDRESS`/`GMAIL_APP_PASSWORD` (+ `OPENROUTER_API_KEY` for
the free triage model), `MAILROOM_GMAIL_ENABLED=1`. Ops laws: one document at a time; a
re-send of the same filename requires quarantining the terminal manifest first (provenance-
aware dedup, HUB-043).


## 1 · Config

Everything the pilot needs in one place. Every knob is env-overridable so the notebook can
be executed headless (`MAILROOM_PILOT_FIRE`, `MAILROOM_PILOT_MODE`, `MAILROOM_PILOT_DOC_CLASS`).
`FIRE` is a deliberate interlock: the default `False` runs the whole notebook as a dry
rehearsal (corpus pick + email build + preflight, no send).

In [1]:
import os
from datetime import datetime, timezone

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
TOKEN = f"pilot{STAMP[-6:]}"

MODE = os.environ.get("MAILROOM_PILOT_MODE", "mock").strip().lower()          # "mock" | "real"
FIRE = os.environ.get("MAILROOM_PILOT_FIRE", "0").strip().lower() in ("1", "true", "yes", "on")
DOC_CLASS = os.environ.get("MAILROOM_PILOT_DOC_CLASS", "insurance_claim").strip() or None
ROLE = os.environ.get("MAILROOM_PILOT_ROLE", "").strip() or None              # snapshot role (e.g. handoff_case_contract)
MATTER_ID = os.environ.get("MAILROOM_PILOT_MATTER", "PILOT-NB").strip()
LIVE_CORPUS = os.environ.get("MAILROOM_HF_LIVE", "0").strip().lower() in ("1", "true", "yes", "on")
MAX_CHARS = int(os.environ.get("MAILROOM_PILOT_MAX_CHARS", "12000"))          # the free-triage input budget
SEED = int(os.environ.get("MAILROOM_PILOT_SEED", "20260904"))
TERMINAL_TIMEOUT_S = float(os.environ.get("MAILROOM_PILOT_TIMEOUT", "600"))

(f"mode={MODE} fire={FIRE} doc_class={DOC_CLASS} role={ROLE} matter={MATTER_ID} "
 f"live_corpus={LIVE_CORPUS} max_chars={MAX_CHARS}")

'mode=mock fire=True doc_class=insurance_claim role=None matter=PILOT-NB live_corpus=False max_chars=12000'

## 2 · Preflight — is the system actually ready?

The pilot is only meaningful against a LIVE watcher: heartbeat fresh, Gmail channel enabled,
free-only guardrail understood (the triage lane runs the free model swarm), and the corpus
source declared. A stale heartbeat here means the FIRE cell would send into a dead room —
fix the watcher first.


In [2]:
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "gmail_pilot_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import gmail_pilot_lab as lab
from IPython.display import display, Markdown

STAMP = lab._now_stamp()
TOKEN = f"pilot{STAMP[-6:]}"

hb = lab.heartbeat_status()
ch = lab.gmail_channel_status()
checks = [
    ("watcher heartbeat fresh", hb.get("fresh"), f"age={hb.get('age_s')}s pid={hb.get('pid')} sha={hb.get('sha')}"),
    ("gmail channel enabled", ch.get("enabled"), f"{ch.get('address')} via {ch.get('smtp_host')}"),
    ("sender allowlist", bool(ch.get("allowed_senders")), f"{len(ch['allowed_senders'])} senders"),
    ("free-only guardrail (pilot posture)", ch.get("free_only_guardrail"), "MAILROOM_LLM_FREE_ONLY"),
    ("corpus source", "live-corpus-load" if LIVE_CORPUS else "committed-snapshot", f"max_chars={MAX_CHARS}"),
]
display(Markdown("| check | ok | detail |\n|---|---|---|\n" + "\n".join(
    f"| {n} | {'yes' if ok else '**NO**'} | {d} |" for n, ok, d in checks)))
assert hb.get("fresh") or not FIRE, "watcher heartbeat is stale — start the watcher before firing"
_ready = True

| check | ok | detail |
|---|---|---|
| watcher heartbeat fresh | yes | age=0.8s pid=37940 sha=8177370e |
| gmail channel enabled | yes | llmmailroom@gmail.com via smtp.gmail.com |
| sender allowlist | yes | 5 senders |
| free-only guardrail (pilot posture) | yes | MAILROOM_LLM_FREE_ONLY |
| corpus source | yes | max_chars=12000 |

## 3 · Pick the pilot document from mailroom-dataset

Offline default: the committed snapshot (one class-representative row per lane plus the
deliberate over-budget handoff case, every row `content_sha256`-verified at snapshot time).
Live mode (`MAILROOM_HF_LIVE=1`) loads the whole split through
`pipeline.hf_corpus_loader` and samples deterministically.


In [3]:
doc = lab.select_document(doc_class=DOC_CLASS, role=ROLE, live=LIVE_CORPUS or None, max_chars=MAX_CHARS, seed=SEED)
prov = doc["provenance"]
labels = doc["labels"]
text = doc["doc_text"]
print(f"source:    {prov['source']}  dataset={prov['dataset']}  sha={prov.get('hub_sha') or prov.get('hub_sha_tip')}")
print(f"filename:  {labels.get('filename')}")
print(f"expected:  {labels.get('expected')} / {labels.get('expected_subclass')}  stage={labels.get('expected_stage')}  intent={labels.get('intent') or '-'}")
print(f"chars:     {len(text)}  (free-triage budget {MAX_CHARS})")
print("-" * 72)
print(text[:600] + ("\n…" if len(text) > 600 else ""))

source:    committed-snapshot  dataset=Lucius-Morningstar/mailroom-corpus  sha=6fc81a8677b072caacd2fc1600492944486c72ea
filename:  carrier:887013387879564.txt
expected:  insurance_claim / carrier  stage=archived  intent=claim_data_record
chars:     1044  (free-triage budget 12000)
------------------------------------------------------------------------
MEDICARE SUMMARY NOTICE -- PHYSICIAN/SUPPLIER CLAIM (Part B)
Notice ID: 887013387879564

Insurer:                  CMS Medicare
Line of business:         health

BENEFICIARY / INSURED
  Insured party:            SANCHEZ, DAVID
  Policy number:            B81C39C31DF18BEB
  Age:                      68
  Sex:                      F
  State/County (SSA):       42 / 030
  Flagged chronic cond.:    diabetes, osteoporosis


SERVICE DETAILS
  Service s
…


## 4 · Build the attachment + the single-document email (dry)

The MIME shape is exactly what the intake poller parses: one attachment (⇒ `route: triage`),
`[M:<matter>]` subject tag, unique `Message-ID`. Nothing is sent yet.


In [4]:
from pathlib import Path

workdir = Path("data/pilot_runs") / f"{STAMP}_{TOKEN}"
attachment_path, attachment_name = lab.build_attachment(doc, workdir, stamp=STAMP)
raw_email, message_id, attachment_name = lab.build_pilot_email(attachment_path, MATTER_ID, stamp=STAMP)
print(f"attachment: {attachment_name}  ({attachment_path.stat().st_size} bytes)")
print(f"message-id: {message_id}")
print(f"subject:    Corpus pilot {STAMP} [M:{MATTER_ID}]")

attachment: pilot_20260904T101734Z_carrier:887013387879564.txt  (1044 bytes)
message-id: <gmail-pilot-3e0352b71e91@mailroom.local>
subject:    Corpus pilot 20260904T101734Z [M:PILOT-NB]


<!-- NB-OPT-IN-NETWORK -->
## 5 · FIRE (interlocked)

Flip `FIRE = True` (or set `MAILROOM_PILOT_FIRE=1`) to actually run the pilot:

- **`MODE="mock"`** — no email at all: the attachment + `route: triage` sidecar land in the
  inbox bin exactly as the poller would deliver them; the live watcher processes it
  end-to-end (free triage lane, echo). Network-free except the LLM call the watcher makes.
- **`MODE="real"`** — the email is REALLY sent via SMTP from the agent mailbox to itself
  (an allowlisted sender) and the IMAP poller sweeps it in — the full Gmail round-trip.


In [5]:
watch = lab.WatchLog()
fired = {"fired": False, "mode": MODE, "message_id": message_id, "at": None}
manifest = None
elapsed_s = 0.0

if FIRE:
    import time as _t
    t0 = _t.time()
    if MODE == "real":
        result = lab.send_via_smtp(raw_email)
        print(f"SMTP sent -> {result['to']} ({result['smtp_host']})")
    elif MODE == "mock":
        result = lab.deliver_mock(attachment_path, MATTER_ID, message_id, stamp=STAMP)
        print(f"inbox drop -> {result['delivered']} (route triage)")
    else:
        raise ValueError(f"unknown MODE {MODE!r}")
    fired.update(fired=True, at=lab._now_stamp())
    manifest, elapsed_s = lab.await_terminal(attachment_name, timeout_s=TERMINAL_TIMEOUT_S)
    watch.drain(attachment_name)
    print(f"terminal manifest: {'yes' if manifest else 'TIMEOUT'} after {elapsed_s:.0f}s"
          + (f" stage={manifest['stage']}" if manifest else ""))
else:
    print("SKIPPED — FIRE interlock is off (dry rehearsal). Flip FIRE or set MAILROOM_PILOT_FIRE=1.")

inbox drop -> pilot_20260904T101734Z_carrier:887013387879564.txt (route triage)


terminal manifest: yes after 30s stage=archived


## 6 · Results — the conveyor, the audit chain, the relations clerk, and the ground truth

Everything the pipeline produced, read from the durable records (manifest, catalog, audit
hash-chain, relations ledger, watcher log) and compared against the corpus row's labels.


In [6]:
import json as _json

evidence = lab.collect_evidence(manifest, watch, token=TOKEN) if FIRE else {"log_events": []}
checks = lab.ground_truth_checks(manifest, doc) if FIRE else []

if manifest:
    triage = ((manifest.get("intake") or {}).get("triage") or {})
    print(f"stage:        {manifest.get('stage')}  (escalation: {manifest.get('escalation_reason') or '-'})")
    print(f"class:        {triage.get('primary_doc_class') or manifest.get('doc_type')} @ {triage.get('confidence') or manifest.get('classification_confidence')}")
    print(f"audit chain:  ok={evidence.get('audit_chain_ok')}  events={evidence.get('audit_events')}")
    print(f"catalog row:  {evidence.get('catalog_row')}")
    print(f"relations:    {len(evidence.get('relations_edges') or [])} edge(s)")
    print(f"extraction:   {_json.dumps(triage.get('extraction') or {}, indent=1)[:800]}")

display(Markdown(("### Ground-truth checks\n\n| check | expected | actual | ok |\n|---|---|---|---|\n"
    + "\n".join(f"| {c['check']} | {c['expected']} | {c['actual']} | {'yes' if c['ok'] else '**NO**'} |" for c in checks))
    if checks else "_Not fired — no ground-truth comparison._"))

2026-09-04 05:18:04 [info     ] schema_ready                   url=sqlite+aiosqlite:////Users/luciusjmorningstar/Downloads/mailroom-dev/packages/llm-mailroom/data/mailroom.db


2026-09-04 05:18:04 [debug    ] relations_context_block_failed


stage:        archived  (escalation: -)
class:        insurance_claim @ 0.95
audit chain:  ok=True  events=['triage_ingested', 'triage_classified', 'triage_archived', 'relations_linked']
catalog row:  {'doc_id': 'cb140526-61f1-4586-9c5d-1e6f83a8d4c5', 'stage': 'archived', 'doc_type': 'insurance_claim', 'doc_subclass': None, 'classification_confidence': 0.95, 'file_sha256': '514adb8d1e66f3aa623b958bce65eee0c053ed4a5dbd388b507069ab77e73eba'}
relations:    9 edge(s)
extraction:   {
 "claim_number": "887013387879564",
 "policy_number": "B81C39C31DF18BEB",
 "insurer": "CMS Medicare",
 "insured_party": "SANCHEZ, DAVID",
 "coverage_determination": "APPROVED"
}


/Users/luciusjmorningstar/Downloads/mailroom-dev/packages/llm-mailroom/src/pipeline/relations.py:908: RuntimeWarning: coroutine 'relations_context' was never awaited
  return ""


### Ground-truth checks

| check | expected | actual | ok |
|---|---|---|---|
| doc_class | insurance_claim | insurance_claim | yes |
| doc_subclass | carrier | None | **NO** |
| stage | archived | archived | yes |
| extraction.claim_number | 887013387879564 | 887013387879564 | yes |
| extraction.policy_number | B81C39C31DF18BEB | B81C39C31DF18BEB | yes |
| extraction.insurer | CMS Medicare | CMS Medicare | yes |
| extraction.insured_party | SANCHEZ, DAVID | SANCHEZ, DAVID | yes |
| extraction.claim_type | health | None | **NO** |
| extraction.date_of_loss | 2009-07-20 | None | **NO** |
| extraction.coverage_determination | approved | APPROVED | yes |

## 7 · Verdict + the durable pilot log

In [7]:
if FIRE and manifest:
    passed = sum(1 for c in checks if c["ok"])
    hard = [c for c in checks if not c.get("soft")]
    verdict = "PASS" if manifest.get("stage") == "archived" and all(c["ok"] for c in hard) else (
        "PASSED WITH SOFT GAPS" if all(c["ok"] for c in hard) else "FAIL — see checks")
    verdict = f"{verdict} ({passed}/{len(checks)} checks ok)"
elif FIRE:
    verdict = "TIMEOUT — no terminal manifest; inspect data/watcher.out + the requeue ops notes below"
else:
    verdict = "DRY REHEARSAL — nothing fired (FIRE interlock off)"

run = {
    "token": TOKEN, "stamp": STAMP, "mode": MODE, "fired": fired,
    "matter_id": MATTER_ID, "attachment_name": attachment_name,
    "doc": {"labels": {k: v for k, v in (doc.get("labels") or {}).items() if k != "doc_text"},
            "provenance": doc.get("provenance"), "chars": len(doc.get("doc_text") or "")},
    "manifest": manifest, "elapsed_s": elapsed_s, "checks": checks,
    "evidence": {k: v for k, v in evidence.items() if k != "log_events"},
    "log_events": (evidence or {}).get("log_events", []), "verdict": verdict,
}
if FIRE:
    paths = lab.write_report(run)
    print(f"report: {paths['json']}\n        {paths['md']}")
print(f"\nVERDICT: {verdict}")

report: /Users/luciusjmorningstar/Downloads/mailroom-dev/packages/llm-mailroom/data/pilot_runs/20260904T101734Z_pilot01734Z/report.json
        /Users/luciusjmorningstar/Downloads/mailroom-dev/packages/llm-mailroom/data/pilot_runs/20260904T101734Z_pilot01734Z/report.md

VERDICT: FAIL — see checks (7/10 checks ok)


## Ops notes

- **Re-queue law (HUB-043):** a re-send of the same filename is refused by the
  provenance-aware dedup until the terminal manifest is quarantined:
  `mv data/manifests/<doc_id>.json data/quarantined_manifests/` then re-fire.
- **Kill switches:** `MAILROOM_GMAIL_TRIAGE=0` (free lane off → full pipeline),
  `MAILROOM_RELATIONS=0`, `MAILROOM_GMAIL_ECHOES=0`, `MAILROOM_WATCHER_STATUS=0`.
- **Where to look when it stalls:** `data/watcher.out` (events), `data/watchdog.out`
  (liveness), `data/debug/triage/` (full free-model I/O for every triage call),
  `data/manifests/` (terminal manifests), `python -m pipeline.relations_scan --verify-ledger`.
- The report under `data/pilot_runs/` is the artifact of record: config, corpus provenance
  (dataset + sha), checks, timeline, verdict.
